In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user may list ingredients in text and/or upload a photo of leftover food.

From a photo, describe what you can see and infer plausible ingredients (say when you are uncertain).

Use the web_search tool to find recipes that fit the ingredients.

Give concise recipe ideas first; share full step-by-step instructions if the user asks.

"""

In [4]:
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

_or_key = os.getenv("OPENROUTER_API_KEY")
_vision_model = os.getenv(
    "OPENROUTER_VISION_MODEL",
    "nvidia/nemotron-nano-12b-v2-vl:free",
)
# Tool calling: pick any OpenRouter model that supports functions; proven in this project.
_chef_model = os.getenv("OPENROUTER_CHEF_MODEL", "openai/gpt-4o-mini")


def _openrouter_llm(model: str) -> ChatOpenAI:
    return ChatOpenAI(
        model=model,
        api_key=_or_key,
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": "https://localhost",
            "X-Title": "lc-foundations",
        },
        temperature=0,
    )


vision_llm = _openrouter_llm(_vision_model)

chef_agent = create_agent(
    model=_openrouter_llm(_chef_model),
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

## Text only: ingredients as a message

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = chef_agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config,
)

print(response["messages"][-1].content)

Here are some quick recipe ideas using your leftover chicken and rice:

1. **Chicken and Rice Casserole**: Combine your leftover chicken and rice with some cream of chicken soup, peas, and cheese. Bake until bubbly for a comforting casserole.

2. **Chicken Fried Rice**: Stir-fry your leftover chicken and rice with vegetables (like peas and carrots), soy sauce, and scrambled eggs for a quick fried rice dish.

3. **Chicken and Rice Skillet**: Sauté the chicken with garlic and onion, add the rice, some broth, and any vegetables you have on hand. Cook until heated through.

4. **Chicken and Rice Soup**: Simmer the chicken and rice in chicken broth with vegetables and seasonings for a hearty soup.

If you'd like detailed instructions for any of these recipes, just let me know!


In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='07a0b5a3-5d8a-4a89-afc5-e2d16fe539fa'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 129, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 3.075e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.075e-05, 'upstream_inference_prompt_cost': 1.935e-05, 'upstream_inference_completions_cost': 1.14e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_d0a1738203', 'id': 'gen-1778195445-C8mEMvB7yLDoWlsqkGXE', 'finish_reason': 'tool_calls', '

## Photo of leftovers (base64, same pattern as `1.4_multimodal_messages.ipynb`)

**OpenRouter:** `vision_llm` runs the VL model on the image; `chef_agent` runs your chef model with `web_search`. Use the exact model IDs from [openrouter.ai/models](https://openrouter.ai/models).

Run upload → encode → invoke.

In [6]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept=".png,.jpg,.jpeg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload')

In [7]:
import base64

if not uploader.value:
    raise ValueError("Upload an image first (run the cell above and pick a file).")

uploaded_file = uploader.value[0]
img_bytes = bytes(uploaded_file["content"])
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

name = uploaded_file.get("name", "").lower()
if name.endswith(".png"):
    mime_type = "image/png"
elif name.endswith((".jpg", ".jpeg")):
    mime_type = "image/jpeg"
else:
    mime_type = "image/jpeg"

In [8]:
from langchain.messages import HumanMessage

config_photo = {"configurable": {"thread_id": "2-photo"}}

# 1) VL model: image → ingredient description (no tools).
multimodal_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "Look at this photo of leftover food. "
                "List ingredients you can identify and note uncertainty. "
                "Do not search the web."
            ),
        },
        {"type": "image", "base64": img_b64, "mime_type": mime_type},
    ]
)
vision_reply = vision_llm.invoke([multimodal_message])

# 2) Agent with web_search: recipes from that description.
followup = HumanMessage(
    content=(
        "From my photo, you summarized:\n\n"
        f"{vision_reply.content}\n\n"
        "Use web_search to find recipes that fit these ingredients. "
        "Give a few options and short reasons."
    )
)
response = chef_agent.invoke({"messages": [followup]}, config_photo)

print(response["messages"][-1].content)

Here are a few recipe ideas that fit your ingredients of salmon fillet, white rice, broccoli, carrot, and grilled lemon:

1. **Lemon Garlic Salmon with Rice & Broccoli**  
   - This recipe features salmon fillets seasoned with lemon and garlic, served alongside fluffy white rice and steamed broccoli. The grilled lemon can enhance the flavor of the salmon and rice.

2. **Grilled Salmon Rice Bowls**  
   - A simple bowl recipe where you can layer cooked white rice, top it with grilled salmon, steamed broccoli, and julienned carrots. Drizzle with a light sauce or lemon juice for added flavor.

3. **Salmon with Broccoli and Rice**  
   - This straightforward recipe combines salmon fillets with broccoli and rice, making it a healthy and balanced meal. You can add a seasoning mix to the salmon for extra flavor.

4. **Rice Bowl with Lemon Herb Garlic Salmon**  
   - This dish includes salmon seasoned with lemon pepper and herbs, served over rice with a side of grilled or steamed vegetables, i